<!-- notebook-header -->
# Vision Transformers e Modelos Modernos

**Modulo:** 05 - Dominios Aplicados / 05A - Computer Vision  
**Tipo:** Aula com exercicios guiados e solucoes executaveis  
**Descricao:** Patches, self-attention visual, ViT, Swin, DETR e modelos vision-language.


# 5A_5: Vision Transformers e Modelos Modernos

## Visao Geral

Neste notebook, exploraremos como Transformers -- originalmente criados para NLP --
revolucionaram visao computacional. Veremos a arquitetura ViT, variantes como Swin e DeiT,
self-supervised learning, CLIP e foundation models.

**Conteudo:**
1. Por que Transformers para Visao
2. Vision Transformer (ViT)
3. Variantes de ViT (DeiT, Swin, MAE)
4. Self-Supervised Learning
5. CLIP e Vision-Language Models
6. Foundation Models (SAM, DINOv2)
7. Exercicios Praticos
8. Erros Comuns

**Pre-requisitos:** 5A_1 (CNN fundamentos), 5A_2 (classificacao)

**Dependencias:** numpy, matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
print('Imports OK')

## 1. Por que Transformers para Visao

### Analogia: Lendo um Documento de 100 Paginas

Imagine que voce precisa entender um documento de 100 paginas:

**CNN (leitura local):** Le cada pagina sequencialmente, captando contexto local.
Para entender como a pagina 3 se relaciona com a pagina 97, precisa
de MUITAS camadas intermediarias propagando informacao.

**Transformer (leitura global):** Olha TODAS as paginas de uma vez. Descobre
que paginas 7 e 45 sao cruciais para a pagina atual. Conecta informacao
distante DIRETAMENTE via self-attention.

### Por que em ML isso importa

CNNs tem **inductive bias local** (kernels so veem vizinhanca). Isso e eficiente
para dados pequenos, mas LIMITA a capacidade de capturar relacoes globais.
Transformers nao assumem localidade -- aprendem QUALQUER padrao se tiverem dados suficientes.

### O Trade-off Fundamental: Dados vs Inductive Bias

| Regime | CNN | ViT |
|--------|-----|-----|
| <10K imagens | Melhor (inductive bias ajuda) | Pior (sem bias, precisa aprender tudo) |
| 10K-1M imagens | Equivalente | Equivalente |
| >1M imagens | Bom | Muito melhor (sem limitacao de bias) |
| >100M imagens | Saturando | Ainda escalando |

### O que observar sobre a Evolucao CNN -> Transformer

A transicao nao foi abrupta. Houve etapas intermediarias:

1. **Attention modules em CNNs** (SE-Net, CBAM) -- 2017-2018
2. **Non-local Neural Networks** (self-attention em features CNN) -- 2018
3. **ViT puro** (sem convolucoes) -- 2020
4. **Hibridos** (ConvNeXt, Swin) voltam a usar ideias convolucionais -- 2021+

### O que concluir sobre quando usar ViT vs CNN

Se voce tem <50K imagens e nao pode usar pre-training, use CNN (ResNet, EfficientNet).
Se pode usar transfer learning de modelos pre-treinados em escala, ViT
geralmente ganha. Na pratica, a maioria dos projetos usa transfer learning,
entao ViT pre-treinado e a escolha padrao em 2024+.

### Conexao com outros notebooks sobre Inductive Bias

O conceito de inductive bias conecta com 2_1 (bias-variance tradeoff) e 4_1
(regularizacao). Mais inductive bias = mais regularizacao implicita = menos
dados necessarios, mas teto de performance mais baixo.

In [ ]:
# Comparacao visual: CNN vs Transformer receptive field
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# CNN: receptive field cresce com camadas
ax = axes[0]
ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.set_title('CNN: Receptive Field Local\n(cresce com profundidade)', fontsize=11, fontweight='bold')

# Grid
for i in range(9):
    ax.axhline(y=i, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i, color='gray', linewidth=0.5, alpha=0.3)

# Center pixel
center = patches.Rectangle((3.5, 3.5), 1, 1, facecolor='red', alpha=0.8)
ax.add_patch(center)

# Layer 1: 3x3
for i in range(3):
    for j in range(3):
        if i == 1 and j == 1:
            continue
        r = patches.Rectangle((2.5+j, 2.5+i), 1, 1, facecolor='orange', alpha=0.4)
        ax.add_patch(r)

# Layer 2: 5x5
for i in range(5):
    for j in range(5):
        x, y = 1.5+j, 1.5+i
        if 2.5 <= x < 5.5 and 2.5 <= y < 5.5:
            continue
        r = patches.Rectangle((x, y), 1, 1, facecolor='yellow', alpha=0.3)
        ax.add_patch(r)

ax.text(4, 0.3, 'Layer 1 (3x3)', color='orange', fontsize=9, ha='center', fontweight='bold')
ax.text(4, -0.2, 'Layer 2 (5x5)', color='goldenrod', fontsize=9, ha='center', fontweight='bold')
ax.set_xlabel('Precisa de MUITAS camadas para contexto global')

# Transformer: global attention from layer 1
ax = axes[1]
ax.set_xlim(0, 8)
ax.set_ylim(0, 8)
ax.set_aspect('equal')
ax.set_title('Transformer: Attention Global\n(desde a primeira camada)', fontsize=11, fontweight='bold')

for i in range(9):
    ax.axhline(y=i, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i, color='gray', linewidth=0.5, alpha=0.3)

center = patches.Rectangle((3.5, 3.5), 1, 1, facecolor='red', alpha=0.8)
ax.add_patch(center)

np.random.seed(42)
for i in range(8):
    for j in range(8):
        if i == 3 and j == 3:
            continue
        alpha = np.random.uniform(0.05, 0.4)
        r = patches.Rectangle((j+0.5, i+0.5), 1, 1, facecolor='blue', alpha=alpha)
        ax.add_patch(r)

ax.set_xlabel('Cada patch "ve" todos os outros diretamente')

# Scaling comparison
ax = axes[2]
data_sizes = [1, 10, 100, 300, 1000]
cnn_acc = [65, 82, 90, 92, 93]
vit_acc = [45, 70, 88, 93, 97]

ax.plot(data_sizes, cnn_acc, 'b-o', linewidth=2, markersize=8, label='CNN (ResNet)')
ax.plot(data_sizes, vit_acc, 'r-s', linewidth=2, markersize=8, label='ViT')
ax.fill_between([1, 30], 0, 100, alpha=0.1, color='blue', label='Regime CNN melhor')
ax.fill_between([30, 1000], 0, 100, alpha=0.1, color='red', label='Regime ViT melhor')
ax.axvline(x=30, color='gray', linestyle='--', alpha=0.5)
ax.text(30, 50, 'Crossover\n~30M imgs', ha='center', fontsize=9, color='gray')

ax.set_xscale('log')
ax.set_xlabel('Milhoes de imagens de treino', fontsize=10)
ax.set_ylabel('Accuracy (%)', fontsize=10)
ax.set_title('Scaling: CNN vs ViT', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(40, 100)

plt.tight_layout()
plt.savefig('/tmp/cnn_vs_transformer.png', dpi=100, bbox_inches='tight')
plt.show()

print('CNN: inductive bias forte -> funciona bem com poucos dados')
print('ViT: sem bias -> precisa de mais dados, mas escala melhor')
print('Crossover: ~14M-30M imagens (ViT original: ImageNet-21K)')

## 2. Vision Transformer (ViT)

### Analogia: Quebra-Cabeca com Etiquetas

Imagine que voce recebe uma foto cortada em 196 pedacos (patches).
Cada pedaco ganha uma etiqueta numerada (positional encoding) para que
voce saiba de onde ele veio. Depois, voce olha TAREFA DO ALUNOS os pedacos ao mesmo
tempo e decide quais sao mais importantes para entender a imagem toda.

### Por que em ML a arquitetura ViT e elegante

ViT e surpreendentemente simples:
1. Corta imagem em patches (como tokens de texto)
2. Projeta cada patch num vetor (embedding)
3. Adiciona posicao (positional encoding)
4. Passa por Transformer Encoder padrao
5. Usa token [CLS] para classificacao

Nenhuma operacao especifica de visao! A mesma arquitetura de NLP funciona.

### Matematica do Patch Embedding

- Imagem: H x W x C (ex: 224 x 224 x 3)
- Patch size: P x P (ex: 16 x 16)
- Numero de patches: N = (H/P) x (W/P) = 196
- Cada patch: P x P x C = 768 valores
- Projecao linear: 768 -> D (dimensao do modelo)
- Sequencia final: [CLS] + 196 patches = 197 tokens

### O que observar sobre Positional Encoding

Sem positional encoding, o Transformer nao sabe onde cada patch esta na imagem.
Poderiamos embaralhar todos os patches e o modelo nao perceberia!
ViT usa positional embeddings APRENDIVEIS (nao sinusoidais como em NLP).

### O que concluir sobre a simplicidade do ViT

A elegancia do ViT e que ele NAO precisa de nenhuma operacao especifica de imagem.
Isso sugere que Transformers sao "learners universais" -- a mesma arquitetura
funciona para texto, imagem, audio, video. A unica coisa que muda e o tokenizer.

In [ ]:

# Visualizar conceito de Vision Transformer
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Imagem original
ax = axes[0, 0]
img = np.random.rand(224, 224, 3)
ax.imshow(img)
ax.set_title('Input Image (224x224x3)', fontsize=12, weight='bold')
ax.axis('off')

# Patch division
ax = axes[0, 1]
ax.imshow(img)
patch_size = 16
num_patches = (224 // patch_size) ** 2
for i in range(0, 224, patch_size):
    ax.axhline(y=i, color='red', linewidth=1, alpha=0.5)
    ax.axvline(x=i, color='red', linewidth=1, alpha=0.5)
ax.set_title(f'Patch Division ({patch_size}x{patch_size})\n{num_patches} patches', 
             fontsize=12, weight='bold')
ax.axis('off')

# Sequence of patches
ax = axes[1, 0]
ax.axis('off')
sequence_text = '''Patch Embedding & Tokenization:

Input: 224x224 image
Patch size: 16x16
Number of patches: (224/16)^2 = 196

Each patch:
  - Flatten 16x16x3 -- > 768 dimensions
  - Project -- > D dimensions (e.g., 768)

Sequence:
  - 196 tokens + 1 [CLS] token
  - Total: 197 tokens
  - Sequence length: 197

Positional Encoding:
  - Learnable positional embeddings
  - Added to patch embeddings
  - Encodes spatial structure
'''
ax.text(0.05, 0.95, sequence_text, transform=ax.transAxes,
       fontsize=10, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Transformer architecture
ax = axes[1, 1]
ax.axis('off')
transformer_text = '''Transformer Encoder Stack:

Input: [cls_token, patch_1, ..., patch_196]
Embedding: (197, D) where D=768

For L layers (e.g., L=12):
  1. Multi-Head Self-Attention
     - Query, Key, Value projections
     - 12 heads for global dependencies
     - Output: (197, D)
  
  2. Layer Normalization
  
  3. Feed-Forward Network
     - 2 layers with GeLU activation
     - Expansion factor: 4
  
  4. Residual connections
     - Add input + output of each block

Output: (197, D)

Classification:
  - Take [CLS] token: (1, D)
  - Linear head -- > num_classes
  - Output: logits for classification
'''
ax.text(0.05, 0.95, transformer_text, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

plt.tight_layout()
plt.savefig('/tmp/vit_architecture.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Demonstracao: como self-attention funciona em patches de imagem
np.random.seed(42)

# Simular 9 patches (3x3 grid) com embeddings de dimensao 4
num_patches = 9
d_model = 4

# Embeddings dos patches (simulados)
patch_embeddings = np.random.randn(num_patches, d_model)

# Simular Query, Key, Value (em ViT real, sao projecoes lineares)
W_q = np.random.randn(d_model, d_model) * 0.5
W_k = np.random.randn(d_model, d_model) * 0.5
W_v = np.random.randn(d_model, d_model) * 0.5

Q = patch_embeddings @ W_q
K = patch_embeddings @ W_k
V = patch_embeddings @ W_v

# Scaled dot-product attention
scale = np.sqrt(d_model)
scores = (Q @ K.T) / scale

# Softmax
def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

attention_weights = softmax(scores)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Attention scores (raw)
ax = axes[0]
im = ax.imshow(scores, cmap='RdBu_r', aspect='equal')
ax.set_title('Attention Scores (raw)\nQ * K^T / sqrt(d)', fontsize=11, fontweight='bold')
ax.set_xlabel('Key (patch)')
ax.set_ylabel('Query (patch)')
for i in range(num_patches):
    for j in range(num_patches):
        ax.text(j, i, f'{scores[i,j]:.1f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)

# Attention weights (after softmax)
ax = axes[1]
im = ax.imshow(attention_weights, cmap='Blues', aspect='equal', vmin=0, vmax=0.3)
ax.set_title('Attention Weights (softmax)\nCada linha soma 1.0', fontsize=11, fontweight='bold')
ax.set_xlabel('Key (patch)')
ax.set_ylabel('Query (patch)')
for i in range(num_patches):
    for j in range(num_patches):
        ax.text(j, i, f'{attention_weights[i,j]:.2f}', ha='center', va='center', fontsize=7)
plt.colorbar(im, ax=ax, shrink=0.8)

# Attention de um patch especifico visualizado na grid
ax = axes[2]
query_patch = 4  # centro da grid 3x3
attn_map = attention_weights[query_patch].reshape(3, 3)
im = ax.imshow(attn_map, cmap='Reds', aspect='equal')
ax.set_title(f'Atencao do Patch {query_patch} (centro)\npara cada outro patch', fontsize=11, fontweight='bold')
for i in range(3):
    for j in range(3):
        val = attn_map[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)

plt.tight_layout()
plt.savefig('/tmp/attention_mechanism.png', dpi=100, bbox_inches='tight')
plt.show()

print('Self-Attention em ViT:')
print(f'  Input: {num_patches} patches, cada um com {d_model} dimensoes')
print(f'  Q, K, V: projecoes lineares dos embeddings')
print(f'  Scores: Q * K^T / sqrt(d) -> shape ({num_patches}, {num_patches})')
print(f'  Weights: softmax(scores) -> cada linha soma 1.0')
print(f'  Output: weights * V -> embeddings atualizados com contexto global')
print()
print('Custo computacional: O(N^2 * d) onde N = numero de patches')
print(f'  Para ViT-Base: N=196, d=768 -> ~29M operacoes por camada de attention')

In [ ]:
# Comparacao de variantes ViT
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Model specs
models = ['ViT-Ti', 'ViT-S', 'ViT-B', 'ViT-L', 'ViT-H']
params = [5.7, 22.1, 86.6, 304.4, 632.0]  # milhoes
layers = [12, 12, 12, 24, 32]
heads = [3, 6, 12, 16, 16]
dims = [192, 384, 768, 1024, 1280]
acc_imagenet = [72.2, 79.9, 81.8, 85.2, 88.6]  # ImageNet-1K with 21K pretrain

# Params vs Accuracy
ax = axes[0]
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(models)))
for i, (m, p, a) in enumerate(zip(models, params, acc_imagenet)):
    ax.scatter(p, a, s=200, c=[colors[i]], zorder=5, edgecolors='black')
    offset_x = 15 if i < 3 else -40
    ax.annotate(f'{m}\n{p:.0f}M', (p, a), fontsize=9, fontweight='bold',
                xytext=(offset_x, 10), textcoords='offset points')

ax.set_xlabel('Parametros (milhoes)', fontsize=10)
ax.set_ylabel('Top-1 Accuracy ImageNet (%)', fontsize=10)
ax.set_title('ViT: Params vs Accuracy\n(pre-treinado ImageNet-21K)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

# Architecture details table
ax = axes[1]
ax.axis('off')
table_data = [['Modelo', 'Layers', 'Heads', 'Dim', 'Params', 'Acc']]
for m, l, h, d, p, a in zip(models, layers, heads, dims, params, acc_imagenet):
    table_data.append([m, str(l), str(h), str(d), f'{p:.1f}M', f'{a:.1f}%'])

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.8)

# Header style
for j in range(len(table_data[0])):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(table_data)):
    for j in range(len(table_data[0])):
        table[i, j].set_facecolor('#D6E4F0' if i % 2 == 0 else 'white')

ax.set_title('Especificacoes das Variantes ViT', fontsize=11, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('/tmp/vit_variants.png', dpi=100, bbox_inches='tight')
plt.show()

print('Observacao: ViT-B (Base) com 86M params e o modelo "padrao"')
print('ViT-L e ViT-H so compensam com pre-training em datasets massivos (>14M imgs)')

## 3. Variantes de ViT

### DeiT: Data-efficient Image Transformers (2021)

**Analogia:** Um aluno (ViT) aprende com um professor experiente (CNN).
O professor ja sabe detectar bordas e texturas. O aluno aprende mais rapido
ao imitar as respostas do professor (knowledge distillation).

### Por que em ML DeiT foi importante

ViT original precisava de ImageNet-21K (14M imagens). DeiT mostrou que com
**distillation token** + augmentation agressiva, ViT funciona com ImageNet-1K (1.3M).
Democratizou ViTs para quem nao tem datasets massivos.

### Swin Transformer (2021)

**Analogia:** Em vez de ler o documento inteiro de uma vez (ViT global attention),
leia por capitulos (windows), mas com paginas que pertencem a dois capitulos
(shifted windows) para manter contexto entre capitulos.

### Por que em ML Swin e popular para deteccao e segmentacao

Swin tem complexidade **linear** (vs quadratica do ViT) porque attention acontece
dentro de windows locais. Alem disso, tem estrutura **hierarquica** (como FPN)
que gera features em multiplas resolucoes -- essencial para deteccao e segmentacao.

### MAE: Masked Autoencoder (2022)

**Analogia:** Jogo de quebra-cabeca onde 75% das pecas estao escondidas.
Se o modelo aprende a reconstruir a imagem completa a partir de 25% dos patches,
ele entendeu profundamente a estrutura visual.

### O que observar sobre a evolucao das variantes

| Modelo | Inovacao | Custo | Dados Minimos | Melhor para |
|--------|----------|-------|---------------|-------------|
| ViT | Patch + Transformer | O(N^2) | 14M+ | Classificacao |
| DeiT | Distillation | O(N^2) | 1.3M | Classificacao |
| Swin | Shifted Windows | O(N) | 1.3M | Deteccao, Segmentacao |
| MAE | Masked pre-training | Baixo | Qualquer | Pre-training |

### O que concluir sobre qual variante escolher

Na pratica em 2024+: use Swin como backbone para deteccao/segmentacao,
ViT para classificacao com transfer learning, e MAE para pre-training
quando voce tem muitos dados nao-rotulados.

### Conexao com outros notebooks sobre Eficiencia

A reducao de O(N^2) para O(N) no Swin conecta com 4_5 (aceleracao hardware).
Menos FLOPs nao so treina mais rapido, mas tambem permite usar imagens
de maior resolucao sem explodir a memoria.

In [ ]:
# Visualizar Swin Transformer windows
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Regular attention (global)
ax = axes[0]
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_aspect('equal')

for i in range(0, 100, 10):
    ax.axhline(y=i, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i, color='gray', linewidth=0.5, alpha=0.3)

ref_patch = plt.Circle((45, 45), 2, color='red', zorder=10)
ax.add_patch(ref_patch)

for i in range(0, 100, 10):
    for j in range(0, 100, 10):
        if not (40 <= i <= 50 and 40 <= j <= 50):
            ax.plot([45, j+5], [45, i+5], 'r-', alpha=0.1, linewidth=0.5)

ax.set_title('Global Attention (ViT)\nO(N^2) complexity', fontsize=11, fontweight='bold')
ax.set_xlabel('Cada patch olha TAREFA DO ALUNOS os outros')
ax.grid(True, alpha=0.2)

# Swin windows (shifted)
ax = axes[1]
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
ax.set_aspect('equal')

for i in range(0, 100, 10):
    ax.axhline(y=i, color='gray', linewidth=0.5, alpha=0.3)
    ax.axvline(x=i, color='gray', linewidth=0.5, alpha=0.3)

# Windows
window_size = 25
window_colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'cyan',
                 'olive', 'magenta', 'teal', 'coral', 'navy', 'lime', 'gold', 'indigo']
idx = 0
for y0 in range(0, 100, 25):
    for x0 in range(0, 100, 25):
        color = window_colors[idx % len(window_colors)]
        rect = patches.Rectangle((x0, y0), window_size, window_size, linewidth=2.5,
                                 edgecolor=color, facecolor=color, alpha=0.08)
        ax.add_patch(rect)
        idx += 1

ref_patch = plt.Circle((45, 45), 2, color='red', zorder=10)
ax.add_patch(ref_patch)

# Attention apenas dentro da window
for i in range(25, 50, 10):
    for j in range(25, 50, 10):
        ax.plot([45, j+5], [45, i+5], 'r-', alpha=0.4, linewidth=1.5)

ax.set_title('Swin Transformer (Shifted Windows)\nO(N) complexity', fontsize=11, fontweight='bold')
ax.set_xlabel('Cada patch olha apenas dentro da window')
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('/tmp/swin_windows.png', dpi=100, bbox_inches='tight')
plt.show()

print('Swin Transformer: Shifted Windows')
print('Vantagens:')
print('  - Complexidade linear: O(N) vs O(N^2) global attention')
print('  - Windows deslocadas: Conexoes entre windows vizinhas')
print('  - Estrutura hierarquica: features em multiplas resolucoes')
print('  - Melhor para deteccao e segmentacao')

In [ ]:
# Visualizacao: conceito do Masked Autoencoder (MAE)
np.random.seed(42)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

# Gerar imagem sintetica com padroes
img = np.zeros((8, 8, 3))
# Gradiente de cor
for i in range(8):
    for j in range(8):
        img[i, j, 0] = i / 7  # vermelho cresce para baixo
        img[i, j, 2] = j / 7  # azul cresce para direita
        img[i, j, 1] = 0.3    # verde constante

# 1. Imagem original
ax = axes[0]
ax.imshow(img, interpolation='nearest')
ax.set_title('Imagem Original\n(64 patches)', fontsize=11, fontweight='bold')
for i in range(9):
    ax.axhline(y=i-0.5, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(x=i-0.5, color='white', linewidth=0.5, alpha=0.5)
ax.axis('off')

# 2. Mascarar 75%
mask = np.random.choice(64, size=48, replace=False)  # 75% masked
masked_img = img.copy()
for idx in mask:
    i, j = idx // 8, idx % 8
    masked_img[i, j] = [0.5, 0.5, 0.5]  # cinza = masked

ax = axes[1]
ax.imshow(masked_img, interpolation='nearest')
ax.set_title('75% Mascarado\n(16 patches visiveis)', fontsize=11, fontweight='bold')
for i in range(9):
    ax.axhline(y=i-0.5, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(x=i-0.5, color='white', linewidth=0.5, alpha=0.5)
ax.axis('off')

# 3. Encoder processa so os 25% visiveis
visible = np.setdiff1d(np.arange(64), mask)
encoder_img = np.ones((8, 8, 3)) * 0.9  # fundo claro
for idx in visible:
    i, j = idx // 8, idx % 8
    encoder_img[i, j] = img[i, j]
    # Highlight
    r = patches.Rectangle((j-0.5, i-0.5), 1, 1, linewidth=2,
                           edgecolor='lime', facecolor='none')
    axes[2].add_patch(r)

ax = axes[2]
ax.imshow(encoder_img, interpolation='nearest')
ax.set_title('Encoder: so 25%\n(4x mais rapido!)', fontsize=11, fontweight='bold')
for i in range(9):
    ax.axhline(y=i-0.5, color='gray', linewidth=0.3, alpha=0.3)
    ax.axvline(x=i-0.5, color='gray', linewidth=0.3, alpha=0.3)
ax.axis('off')

# 4. Decoder reconstroi tudo
# Simular reconstrucao imperfeita (com pequeno ruido)
recon = img.copy() + np.random.randn(8, 8, 3) * 0.05
recon = np.clip(recon, 0, 1)

ax = axes[3]
ax.imshow(recon, interpolation='nearest')
ax.set_title('Decoder: Reconstroi\n(preve pixels mascarados)', fontsize=11, fontweight='bold')
for i in range(9):
    ax.axhline(y=i-0.5, color='white', linewidth=0.5, alpha=0.5)
    ax.axvline(x=i-0.5, color='white', linewidth=0.5, alpha=0.5)
ax.axis('off')

plt.suptitle('Masked Autoencoder (MAE): Mascarar 75%, Reconstruir Tudo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/mae_concept.png', dpi=100, bbox_inches='tight')
plt.show()

print('MAE: eficiencia computacional')
print(f'  Encoder processa apenas 25% dos patches (4x menos compute)')
print(f'  Decoder (leve) reconstroi os 75% mascarados')
print(f'  Pre-training: 1600 epochs em ImageNet em tempo razoavel')
print(f'  Resultado: representacoes tao boas quanto contrastive learning')

## 4. Self-Supervised Learning para Visao

### Analogia: Aprender Idioma por Imersao

Self-supervised learning e como aprender um idioma morando no pais:
- **Ninguem te da labels** (nao tem professor dizendo "isso e um cachorro")
- **Voce aprende pelo contexto** (se uma palavra aparece com certas imagens, voce infere o significado)
- **Quanto mais exposicao, melhor** (mais dados = melhor representacao)

### Por que em ML self-supervised e revolucionario

Labels sao CAROS. ImageNet custou milhoes de dolares para anotar.
Self-supervised learning usa os PROPRIOS dados como supervisao:
- **Contrastive:** "essas duas augmentacoes da mesma imagem devem ter embeddings similares"
- **Masked:** "reconstrua os patches que eu escondi"
- **Distillation:** "a saida do student deve ser parecida com a do teacher"

### As Tres Familias de Self-Supervised Learning

1. **Contrastive (SimCLR, MoCo, BYOL):** Pares positivos/negativos no espaco de embedding
2. **Masked (MAE, BEiT):** Reconstruir partes mascaradas da entrada
3. **Self-Distillation (DINO):** Student imita teacher sem labels

### O que observar sobre o impacto pratico

Self-supervised pre-training + fine-tuning com poucos labels frequentemente
SUPERA supervised pre-training. Com apenas 1% dos labels do ImageNet,
DINO fine-tuned atinge ~75% accuracy (vs ~60% de um ViT treinado from scratch).

### O que concluir sobre o futuro do supervised learning

Supervised learning nao vai morrer, mas se tornara a etapa de fine-tuning.
Pre-training sera cada vez mais self-supervised (sem custo de anotacao).
Isso muda a economia de ML: dados abundantes nao-rotulados > poucos dados rotulados.

### Conexao com outros notebooks sobre Aprendizado Nao-Supervisionado

Self-supervised conecta com 3_3 (clustering, representacoes) -- ambos aprendem
estrutura dos dados sem labels. A diferenca e que self-supervised gera
representacoes mais uteis para downstream tasks que clustering classico.

In [ ]:

# Visualizar contrastive learning
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Augmentação de imagem
ax = axes[0]
original = np.random.rand(100, 100, 3)
ax.imshow(original)
ax.set_title('Original Image', fontsize=11, weight='bold')
ax.axis('off')

# Augmentação 1
ax = axes[1]
aug1 = np.roll(np.roll(original, 10, axis=0), 15, axis=1)  # Shift simulado
ax.imshow(aug1)
ax.set_title('Augmentation 1\n(Crop, Flip, Color Jitter)', fontsize=11, weight='bold')
ax.axis('off')

# Espaço de embedding
ax = axes[2]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 2)
ax.set_aspect('equal')

# Embeddings da mesma imagem (aumentadas)
ax.plot([0.2, 0.3], [0.2, 0.25], 'go-', linewidth=2, markersize=10, label='Same image embeddings')
ax.text(0.4, 0.25, 'Minimize distance', fontsize=9)

# Embeddings de outras imagens
ax.plot([1.5, 1.6], [1.5, 1.4], 'rx', markersize=10, label='Other images')
ax.plot([1.4, 1.3], [0.8, 0.9], 'rx', markersize=10)
ax.text(1.7, 1.5, 'Maximize distance', fontsize=9)

# Similaridade
circle = plt.Circle((0.25, 0.22), 0.5, fill=False, edgecolor='green', linewidth=2, linestyle='--')
ax.add_patch(circle)

ax.set_xlabel('Embedding Dimension 1')
ax.set_ylabel('Embedding Dimension 2')
ax.set_title('Embedding Space\n(Contrastive Learning)', fontsize=11, weight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/contrastive_learning.png', dpi=100, bbox_inches='tight')
plt.show()

print('Contrastive Learning Framework:')
print('=' * 50)
print()
print('SimCLR (Simple Framework for Contrastive Learning):')
print('  1. Aplicar 2 augmentações -- > x_i e x_j')
print('  2. Encoder: f(x) -- > embedding h')
print('  3. Projection: g(h) -- > z (normalized)')
print('  4. Loss: Maximize sim(z_i, z_j) / sim(z_i, z_k) para k != j')
print()
print('MoCo (Momentum Contrast):')
print('  1. Encoders: query encoder + momentum encoder')
print('  2. Queue: Large buffer de negative samples')
print('  3. Momentum update: Smooth update do encoder')
print('  4. Vantagem: Consistência melhor com less memory')


In [ ]:
# Demonstracao: pipeline de contrastive learning (SimCLR simplificado)
np.random.seed(42)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Gerar "imagens" sinteticas (patches de cor)
n_images = 6
images = np.random.rand(n_images, 4, 4, 3)

# Simular embeddings (encoder + projection head)
# Augmentacoes da mesma imagem devem ter embeddings proximos
embeddings = []
labels = []  # qual imagem original
for img_idx in range(n_images):
    # Embedding base + ruido pequeno (augmentacao 1)
    base = np.random.randn(2) * 0.3
    base[0] += img_idx * 0.7  # separar clusters
    base[1] += (img_idx % 3) * 0.5
    embeddings.append(base + np.random.randn(2) * 0.1)
    labels.append(img_idx)
    # Augmentacao 2 (proxima da 1)
    embeddings.append(base + np.random.randn(2) * 0.1)
    labels.append(img_idx)

embeddings = np.array(embeddings)
labels = np.array(labels)

# Row 1: Augmentation pipeline
ax = axes[0, 0]
ax.imshow(images[0], interpolation='nearest')
ax.set_title('Imagem Original', fontsize=11, fontweight='bold')
ax.axis('off')

ax = axes[0, 1]
aug1 = np.flip(images[0], axis=1)  # flip horizontal
ax.imshow(aug1, interpolation='nearest')
ax.set_title('Augmentacao 1\n(flip + crop)', fontsize=11, fontweight='bold')
ax.axis('off')

ax = axes[0, 2]
aug2 = images[0] * 0.7 + 0.15  # color jitter simulado
ax.imshow(np.clip(aug2, 0, 1), interpolation='nearest')
ax.set_title('Augmentacao 2\n(color jitter)', fontsize=11, fontweight='bold')
ax.axis('off')

# Row 2: Embedding space
ax = axes[1, 0]
colors_map = plt.cm.tab10(np.linspace(0, 1, n_images))
for i in range(n_images):
    mask = labels == i
    ax.scatter(embeddings[mask, 0], embeddings[mask, 1], c=[colors_map[i]],
              s=100, label=f'Img {i}', edgecolors='black', linewidth=0.5)
    # Linha conectando par positivo
    pts = embeddings[mask]
    ax.plot(pts[:, 0], pts[:, 1], '-', color=colors_map[i], alpha=0.5, linewidth=2)

ax.set_title('Espaco de Embedding\n(pares positivos conectados)', fontsize=11, fontweight='bold')
ax.set_xlabel('Dim 1')
ax.set_ylabel('Dim 2')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)

# Similarity matrix
ax = axes[1, 1]
n = len(embeddings)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        # Cosine similarity
        dot = np.dot(embeddings[i], embeddings[j])
        norm = np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j])
        sim_matrix[i, j] = dot / (norm + 1e-8)

im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_title('Matriz de Similaridade\n(diagonal = pares positivos)', fontsize=11, fontweight='bold')
ax.set_xlabel('Sample')
ax.set_ylabel('Sample')
plt.colorbar(im, ax=ax, shrink=0.8)

# Contrastive loss explanation
ax = axes[1, 2]
ax.axis('off')
loss_text = (
    "NT-Xent Loss (SimCLR):\n\n"
    "Para cada par positivo (i, j):\n\n"
    "L(i,j) = -log( exp(sim(i,j)/T) /\n"
    "               sum(exp(sim(i,k)/T)) )\n\n"
    "Onde:\n"
    "- sim(i,j) = cosseno entre embeddings\n"
    "- T = temperatura (0.07 tipico)\n"
    "- k percorre todos os samples\n"
    "  (exceto i)\n\n"
    "Intuitivamente:\n"
    "- MAXIMIZE similaridade do par positivo\n"
    "- MINIMIZE similaridade com negativos\n"
    "- Temperatura controla sharpness"
)
ax.text(0.05, 0.95, loss_text, transform=ax.transAxes,
        fontsize=9, verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('/tmp/simclr_pipeline.png', dpi=100, bbox_inches='tight')
plt.show()

print('SimCLR key insight: augmentations definem o que e "a mesma coisa"')
print('Augmentacoes agressivas forcam o modelo a aprender semantica, nao textura')

## 5. CLIP e Vision-Language Models

### Analogia: Aprender com Legendas de Fotos

CLIP aprende como uma crianca que ve fotos com legendas:
- Ve foto de cachorro + texto "um cachorro no parque"
- Aprende que aquela IMAGEM corresponde a aquele TEXTO
- Com milhoes de pares, desenvolve entendimento semantico profundo
- Depois, dado QUALQUER texto, sabe qual imagem corresponde (e vice-versa)

### Por que em ML CLIP mudou o paradigma

Antes do CLIP, cada tarefa precisava de labels especificos:
- Classificacao: labels por classe
- Deteccao: bounding boxes anotados
- Segmentacao: mascaras pixel-level

CLIP aprende com TEXT-IMAGE PAIRS (disponivel na internet em bilhoes!).
400 milhoes de pares, coletados automaticamente, custando ~zero de anotacao humana.

### Zero-Shot: Classificar sem Treinar

1. Encode imagem -> embedding de imagem
2. Encode textos "a photo of a {classe}" -> embeddings de texto
3. Similaridade cosseno entre imagem e cada texto
4. Classe = texto com maior similaridade

Nao precisa NENHUM exemplo da classe durante treino!

### O que observar sobre os limites de CLIP

CLIP nao e perfeito:
- Falha em contagem ("3 gatos" vs "5 gatos")
- Falha em relacoes espaciais ("gato em cima da mesa")
- Falha em conceitos abstratos nao visuais
- Bias dos dados de treino (textos da internet)

### O que concluir sobre Vision-Language como paradigma

Vision-Language models (CLIP, ALIGN, SigLIP) representam uma mudanca fundamental:
de "treinar classificador para N classes" para "alinhar visao e linguagem no
mesmo espaco semantico". Isso e a base de DALL-E, Stable Diffusion, GPT-4V.

### Conexao com outros notebooks sobre Transfer Learning

CLIP e o transfer learning levado ao extremo (5A_2). Em vez de pre-treinar
em ImageNet (1K classes), pre-treina com linguagem natural (vocabulario ilimitado).
O "vocabulary" de conceitos e infinito.

In [ ]:

# Visualizar conceito de CLIP
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Image encoder
ax = axes[0, 0]
ax.axis('off')
image_text = '''Image Encoder (ViT):

Input: Image (224x224x3)

Process:
  1. Patch embedding
  2. Transformer layers (L)
  3. Global average pooling
  4. L2 normalization

Output: 
  - Image embedding (512-dim)
  - Same space as text embedding
'''
ax.text(0.05, 0.95, image_text, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7))

# Text encoder
ax = axes[0, 1]
ax.axis('off')
text_enc = '''Text Encoder (Transformer):

Input: Text prompt
  "A photo of a dog"

Process:
  1. Token embedding + positional
  2. Transformer layers (L)
  3. Take [EOS] token representation
  4. L2 normalization

Output:
  - Text embedding (512-dim)
  - Same space as image embedding
'''
ax.text(0.05, 0.95, text_enc, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

# Contrastive loss
ax = axes[1, 0]
ax.axis('off')
loss_text = '''Contrastive Loss:

For batch of N (image, text) pairs:

1. Compute similarity matrix:
   S[i,j] = dot(image_embed[i], text_embed[j])

2. Create labels:
   - S[i,i] should be high (1)
   - S[i,j] should be low (0) for i != j

3. Loss:
   - Row-wise CE: Prediz correta image
   - Column-wise CE: Prediz correto text

4. Optimize jointly com single loss
'''
ax.text(0.05, 0.95, loss_text, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

# Zero-shot classification
ax = axes[1, 1]
ax.axis('off')
zero_shot = '''Zero-Shot Classification:

Test Image: Dog photo

Text prompts:
  - "A photo of a dog"
  - "A photo of a cat"
  - "A photo of a bird"

Similarity scores:
  - dog: 0.82
  - cat: 0.15
  - bird: 0.03

Prediction: dog (argmax)

Vantagens:
  ✓ No fine-tuning needed
  ✓ Add classes without retraining
  ✓ Generalize to new domains
  ✓ Semantic understanding
'''
ax.text(0.05, 0.95, zero_shot, transform=ax.transAxes,
       fontsize=9, verticalalignment='top', family='monospace',
       bbox=dict(boxstyle='round', facecolor='lightcoral', alpha=0.7))

plt.tight_layout()
plt.savefig('/tmp/clip_concept.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Demonstracao: Zero-Shot Classification conceitual
np.random.seed(42)

# Simular embeddings de CLIP (512-dim na realidade, usamos 2D para visualizacao)
n_classes = 5
class_names = ['cachorro', 'gato', 'carro', 'aviao', 'flor']
class_prompts = [f'a photo of a {c}' for c in class_names]

# Embeddings de texto (simulados como clusters no 2D)
text_embeddings = np.array([
    [1.0, 0.3],   # cachorro
    [0.8, 0.5],   # gato (proximo de cachorro - ambos animais)
    [-0.5, -0.8],  # carro
    [-0.3, 0.9],   # aviao
    [0.2, -0.6],   # flor
])
# Normalizar
for i in range(len(text_embeddings)):
    text_embeddings[i] /= np.linalg.norm(text_embeddings[i])

# Imagem de teste: um cachorro
# Embedding proximo do texto "cachorro"
test_image_embed = np.array([0.95, 0.35])
test_image_embed /= np.linalg.norm(test_image_embed)

# Calcular similaridades
similarities = text_embeddings @ test_image_embed
probs = np.exp(similarities * 10) / np.sum(np.exp(similarities * 10))  # softmax com temp

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Embedding space
ax = axes[0]
colors = ['brown', 'orange', 'blue', 'gray', 'green']
for i, (name, emb, color) in enumerate(zip(class_names, text_embeddings, colors)):
    ax.scatter(emb[0], emb[1], s=200, c=color, marker='s', edgecolors='black',
              linewidth=1.5, zorder=5)
    ax.annotate(f'"{class_prompts[i]}"', (emb[0], emb[1]), fontsize=8,
                xytext=(10, 10), textcoords='offset points')

ax.scatter(test_image_embed[0], test_image_embed[1], s=300, c='red', marker='*',
          edgecolors='black', linewidth=1.5, zorder=10, label='Test image (cachorro)')

# Linhas de similaridade
for i, emb in enumerate(text_embeddings):
    alpha = max(0.1, probs[i])
    ax.plot([test_image_embed[0], emb[0]], [test_image_embed[1], emb[1]],
            '--', color=colors[i], alpha=alpha, linewidth=2)

ax.set_xlabel('Embedding Dim 1', fontsize=10)
ax.set_ylabel('Embedding Dim 2', fontsize=10)
ax.set_title('Espaco de Embedding CLIP\n(imagem e textos no mesmo espaco)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Probabilidades zero-shot
ax = axes[1]
bars = ax.barh(class_names, probs, color=colors, edgecolor='black', linewidth=0.8)
ax.set_xlabel('Probabilidade', fontsize=10)
ax.set_title('Zero-Shot Classification\n(sem treino nesta tarefa!)', fontsize=11, fontweight='bold')
for i, (p, s) in enumerate(zip(probs, similarities)):
    ax.text(p + 0.01, i, f'{p:.1%} (sim={s:.2f})', va='center', fontsize=9)
ax.set_xlim(0, max(probs) * 1.4)

plt.tight_layout()
plt.savefig('/tmp/clip_zero_shot.png', dpi=100, bbox_inches='tight')
plt.show()

print('Zero-Shot CLIP:')
print(f'  Imagem de teste: cachorro')
print(f'  Predicao: {class_names[np.argmax(probs)]} ({probs[np.argmax(probs)]:.1%})')
print()
print('Sem NENHUM exemplo de treino para essas classes!')
print('Basta descrever a classe em linguagem natural.')

## 6. Foundation Models para Visao

### Analogia: Enciclopedia Visual Universal

Foundation models sao como uma enciclopedia que sabe TUDO sobre imagens:
- **SAM (Segment Anything):** "me mostre qualquer objeto e eu segmento"
- **DINOv2:** "me de qualquer imagem e eu extraio features uteis"
- **GPT-4V/Gemini:** "me de qualquer imagem e eu descrevo em linguagem natural"

### Por que em ML foundation models sao transformadores

Foundation models mudam a economia de ML:
- **Antes:** treinar modelo do zero para cada tarefa (caro, lento, precisa de dados)
- **Depois:** usar foundation model + prompt/fine-tune minimo (rapido, barato, pouco dado)

### O que observar sobre SAM (Segment Anything Model)

SAM foi treinado com 1.1 BILHAO de mascaras em 11M imagens.
Aceita prompts: pontos, bounding boxes, texto.
Zero-shot: segmenta objetos NUNCA vistos durante treino.
Limitacao: nao sabe o que o objeto E (segmenta mas nao classifica).

### O que concluir sobre o paradigma de Foundation Models

Estamos na transicao de "um modelo por tarefa" para "um modelo para tudo".
Na pratica, o workflow em 2024+ e:
1. Escolha um foundation model (DINOv2, CLIP, SAM)
2. Extraia features ou use como backbone
3. Fine-tune apenas a cabeca para sua tarefa especifica
4. Deploy

### Conexao com outros notebooks sobre Transfer Learning

Foundation models sao a extensao natural de transfer learning (5A_2).
A diferenca e escala: pre-training em bilhoes de exemplos vs milhoes.
Quanto maior o pre-training, menos fine-tuning voce precisa.

In [ ]:
# Visualizacao: comparacao de Foundation Models
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Timeline de foundation models
ax = axes[0]
ax.set_xlim(2019.5, 2024.5)
ax.set_ylim(-0.5, 7.5)

models_timeline = [
    (2020.0, 0, 'ViT', 'Patch + Transformer', '#4CAF50'),
    (2021.0, 1, 'CLIP', '400M image-text pairs', '#2196F3'),
    (2021.0, 2, 'DINO', 'Self-distillation', '#FF9800'),
    (2021.5, 3, 'Swin', 'Shifted windows', '#9C27B0'),
    (2022.0, 4, 'MAE', 'Masked autoencoder', '#F44336'),
    (2023.0, 5, 'SAM', '1.1B masks', '#00BCD4'),
    (2023.0, 6, 'DINOv2', 'Curated data + iBOT', '#FF5722'),
    (2023.5, 7, 'SAM 2', 'Video segmentation', '#607D8B'),
]

for year, y, name, desc, color in models_timeline:
    ax.barh(y, 0.3, left=year-0.15, height=0.6, color=color, alpha=0.7, edgecolor='black')
    ax.text(year, y, name, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    ax.text(year + 0.25, y, f'  {desc}', ha='left', va='center', fontsize=8)

ax.set_xlabel('Ano', fontsize=10)
ax.set_title('Timeline de Foundation Models\npara Visao Computacional', fontsize=11, fontweight='bold')
ax.set_yticks([])
ax.grid(True, axis='x', alpha=0.3)

# Capabilities comparison
ax = axes[1]
ax.axis('off')

capabilities = [
    ['Modelo', 'Classificacao', 'Deteccao', 'Segmentacao', 'Zero-Shot', 'Dados Treino'],
    ['ViT', 'Excelente', 'Bom*', 'Bom*', 'Nao', '14M labels'],
    ['CLIP', 'Bom', 'Nao', 'Nao', 'Sim', '400M pares'],
    ['DINO/v2', 'Excelente', 'Excelente', 'Excelente', 'Parcial', '142M imgs'],
    ['SAM', 'Nao', 'Nao', 'Excelente', 'Sim', '11M + 1.1B masks'],
    ['MAE', 'Bom', 'Bom', 'Bom', 'Nao', 'Self-supervised'],
]

table = ax.table(cellText=capabilities, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.8)

for j in range(len(capabilities[0])):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, len(capabilities)):
    for j in range(len(capabilities[0])):
        table[i, j].set_facecolor('#D6E4F0' if i % 2 == 0 else 'white')

ax.set_title('Capacidades dos Foundation Models\n(*requer adaptacao)', fontsize=11, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('/tmp/foundation_models.png', dpi=100, bbox_inches='tight')
plt.show()

print('Foundation models: a tendencia e clara')
print('  2020: modelos por tarefa')
print('  2023: modelos universais + fine-tuning minimo')
print('  2024+: modelos multimodais (visao + texto + audio)')

## 7. Exercicios Praticos

### Exercicio 1: Self-Attention Manual

Implemente self-attention de forma manual com numpy para entender o mecanismo.

**Tarefa:** Dados patches simulados, calcule Q, K, V, scores e attention weights.
Verifique que cada linha da matriz de pesos soma 1.

In [ ]:
# PRATICA - Exercicio 1: Self-Attention Manual
# Implemente self-attention passo a passo

np.random.seed(123)
num_patches = 4
d_model = 3

# Embeddings dos patches
X = np.random.randn(num_patches, d_model)

# Matrizes de projecao (pesos)
W_q = np.random.randn(d_model, d_model) * 0.5
W_k = np.random.randn(d_model, d_model) * 0.5
W_v = np.random.randn(d_model, d_model) * 0.5

# TAREFA DO ALUNO: Calcule Q, K, V
Q = None  # X @ W_q
K = None  # X @ W_k
V = None  # X @ W_v

# TAREFA DO ALUNO: Calcule attention scores (Q * K^T / sqrt(d))
scores = None

# TAREFA DO ALUNO: Aplique softmax em cada linha
def softmax_rows(x):
    # TAREFA DO ALUNO: implementar softmax por linha
    return None

attn_weights = None  # softmax_rows(scores)

# TAREFA DO ALUNO: Calcule output (weights @ V)
output = None

# Verificacoes
print('Self-Attention Manual')
print(f'X shape: {X.shape}')
if Q is not None:
    print(f'Q shape: {Q.shape}')
    print(f'Scores shape: {scores.shape}')
    print(f'Weights shape: {attn_weights.shape}')
    print(f'Soma de cada linha: {attn_weights.sum(axis=1)}')  # deve ser [1, 1, 1, 1]
    print(f'Output shape: {output.shape}')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 1: Self-Attention Manual
np.random.seed(123)
num_patches = 4
d_model = 3

X = np.random.randn(num_patches, d_model)
W_q = np.random.randn(d_model, d_model) * 0.5
W_k = np.random.randn(d_model, d_model) * 0.5
W_v = np.random.randn(d_model, d_model) * 0.5

# Projecoes Q, K, V
Q = X @ W_q
K = X @ W_k
V = X @ W_v

# Scores = Q * K^T / sqrt(d)
scale = np.sqrt(d_model)
scores = (Q @ K.T) / scale

# Softmax por linha
def softmax_rows(x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x / np.sum(e_x, axis=1, keepdims=True)

attn_weights = softmax_rows(scores)

# Output = weights @ V
output = attn_weights @ V

print('Self-Attention Manual - SOLUCAO')
print(f'X shape: {X.shape}')
print(f'Q shape: {Q.shape}, K shape: {K.shape}, V shape: {V.shape}')
print(f'Scores shape: {scores.shape}')
print()
print('Attention Weights (cada linha soma 1):')
for i in range(num_patches):
    row = ' '.join(f'{w:.3f}' for w in attn_weights[i])
    print(f'  Patch {i}: [{row}] soma={attn_weights[i].sum():.4f}')
print()
print(f'Output shape: {output.shape}')
print('Cada patch agora contem informacao ponderada de TAREFA DO ALUNOS os outros patches')

### Exercicio 2: Efeito do Positional Encoding

**Tarefa:** Mostre que sem positional encoding, embaralhar os patches
nao muda o resultado da attention. Com positional encoding, o resultado muda.
Isso prova que positional encoding e essencial para informacao espacial.

In [ ]:
# PRATICA - Exercicio 2: Positional Encoding
np.random.seed(42)
num_patches = 4
d_model = 4

# Patches originais
patches_original = np.random.randn(num_patches, d_model)

# Ordem embaralhada
shuffle_idx = [2, 0, 3, 1]
patches_shuffled = patches_original[shuffle_idx]

# TAREFA DO ALUNO: Criar positional encoding (um vetor diferente para cada posicao)
pos_encoding = None  # np.random.randn(num_patches, d_model) * 0.1

# TAREFA DO ALUNO: Adicionar positional encoding aos patches
patches_with_pos = None        # patches_original + pos_encoding
patches_shuffled_with_pos = None  # patches_shuffled + pos_encoding

# TAREFA DO ALUNO: Calcular attention scores SEM positional encoding
# (use patches_original e patches_shuffled diretamente)
W = np.random.randn(d_model, d_model) * 0.3

scores_original_no_pos = None
scores_shuffled_no_pos = None

# TAREFA DO ALUNO: Calcular attention scores COM positional encoding
scores_original_with_pos = None
scores_shuffled_with_pos = None

# Comparar
if scores_original_no_pos is not None:
    diff_no_pos = np.abs(scores_original_no_pos - scores_shuffled_no_pos).mean()
    diff_with_pos = np.abs(scores_original_with_pos - scores_shuffled_with_pos).mean()
    print(f'Diferenca SEM pos encoding: {diff_no_pos:.6f}')
    print(f'Diferenca COM pos encoding: {diff_with_pos:.6f}')
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 2: Positional Encoding
np.random.seed(42)
num_patches = 4
d_model = 4

patches_original = np.random.randn(num_patches, d_model)
shuffle_idx = [2, 0, 3, 1]
patches_shuffled = patches_original[shuffle_idx]

# Positional encoding: vetor unico por posicao
pos_encoding = np.random.randn(num_patches, d_model) * 0.3

# Com positional encoding
patches_with_pos = patches_original + pos_encoding
patches_shuffled_with_pos = patches_shuffled + pos_encoding  # mesma pos, patches diferentes

# Projecao simples
W = np.random.randn(d_model, d_model) * 0.3

# SEM positional encoding: Q*K^T
Q_orig = patches_original @ W
K_orig = patches_original @ W
scores_orig_no_pos = Q_orig @ K_orig.T / np.sqrt(d_model)

Q_shuf = patches_shuffled @ W
K_shuf = patches_shuffled @ W
scores_shuf_no_pos = Q_shuf @ K_shuf.T / np.sqrt(d_model)

# COM positional encoding
Q_orig_pos = patches_with_pos @ W
K_orig_pos = patches_with_pos @ W
scores_orig_with_pos = Q_orig_pos @ K_orig_pos.T / np.sqrt(d_model)

Q_shuf_pos = patches_shuffled_with_pos @ W
K_shuf_pos = patches_shuffled_with_pos @ W
scores_shuf_with_pos = Q_shuf_pos @ K_shuf_pos.T / np.sqrt(d_model)

print('Efeito do Positional Encoding - SOLUCAO')
print()

# SEM pos: scores sao permutacao um do outro (mesma informacao)
print('SEM Positional Encoding:')
print(f'  Scores original diagonal: {np.diag(scores_orig_no_pos).round(3)}')
print(f'  Scores shuffled diagonal: {np.diag(scores_shuf_no_pos).round(3)}')
print(f'  (scores sao permutacao um do outro -- nao ha info de posicao)')
print()

# COM pos: scores sao DIFERENTES (posicao importa)
print('COM Positional Encoding:')
print(f'  Scores original diagonal: {np.diag(scores_orig_with_pos).round(3)}')
print(f'  Scores shuffled diagonal: {np.diag(scores_shuf_with_pos).round(3)}')
diff = np.abs(scores_orig_with_pos - scores_shuf_with_pos).mean()
print(f'  Diferenca media: {diff:.4f}')
print(f'  (scores DIFERENTES -- posicao afeta o resultado!)')
print()
print('Conclusao: sem positional encoding, o Transformer trata patches como BAG OF PATCHES')
print('Com positional encoding, a POSICAO de cada patch influencia a attention')

### Exercicio 3: Padroes de Attention

**Tarefa:** Simule uma imagem 4x4 com um objeto (foreground) e fundo (background).
Mostre que attention de patches do objeto tende a focar em outros patches do objeto,
nao no fundo. Isso ilustra como ViT aprende a "agrupar" regioes semanticas.

In [ ]:
# PRATICA - Exercicio 3: Padroes de Attention Semantica
np.random.seed(42)

# Imagem 4x4 com objeto no centro
# Patches do objeto tem valores altos, fundo tem valores baixos
grid_size = 4
d_model = 4
n_patches = grid_size * grid_size

# TAREFA DO ALUNO: Criar embeddings onde patches do "objeto" sao similares entre si
# e diferentes dos patches do "fundo"
# Objeto: patches [5, 6, 9, 10] (centro 2x2 de grid 4x4)
object_patches = {5, 6, 9, 10}
background_patches = set(range(16)) - object_patches

# TAREFA DO ALUNO: Gerar embeddings
embeddings = None  # np.zeros((n_patches, d_model))
# Para patches do objeto: vetor base_obj + ruido pequeno
# Para patches do fundo: vetor base_bg + ruido pequeno

# TAREFA DO ALUNO: Calcular attention weights
# attn = softmax(embeddings @ embeddings.T / sqrt(d))

# TAREFA DO ALUNO: Visualizar attention de um patch do objeto (ex: patch 5)
# Mostrar que ele atende mais aos patches 6, 9, 10 (outros do objeto)

if embeddings is not None:
    print('Attention do patch 5 (objeto):')
    # Mostrar os pesos
else:
    print('Complete os TAREFA DO ALUNOs acima!')

In [ ]:
# SOLUCAO - Exercicio 3: Padroes de Attention Semantica
np.random.seed(42)

grid_size = 4
d_model = 4
n_patches = grid_size * grid_size
object_patches = {5, 6, 9, 10}

# Embeddings: objeto e fundo como clusters separados
base_obj = np.array([1.0, 0.5, -0.3, 0.8])
base_bg = np.array([-0.5, -0.8, 0.7, -0.2])

embeddings = np.zeros((n_patches, d_model))
for i in range(n_patches):
    if i in object_patches:
        embeddings[i] = base_obj + np.random.randn(d_model) * 0.2
    else:
        embeddings[i] = base_bg + np.random.randn(d_model) * 0.2

# Attention weights
scores = embeddings @ embeddings.T / np.sqrt(d_model)
e_scores = np.exp(scores - np.max(scores, axis=1, keepdims=True))
attn_weights = e_scores / np.sum(e_scores, axis=1, keepdims=True)

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Grid com objeto marcado
ax = axes[0]
grid = np.zeros((grid_size, grid_size))
for idx in object_patches:
    grid[idx // grid_size, idx % grid_size] = 1.0
ax.imshow(grid, cmap='RdYlGn', vmin=-0.5, vmax=1.5)
for i in range(grid_size):
    for j in range(grid_size):
        idx = i * grid_size + j
        label = 'OBJ' if idx in object_patches else 'BG'
        ax.text(j, i, f'{idx}\n{label}', ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_title('Imagem 4x4\n(objeto no centro)', fontsize=11, fontweight='bold')
ax.axis('off')

# 2. Attention do patch 5 (objeto)
ax = axes[1]
attn_5 = attn_weights[5].reshape(grid_size, grid_size)
im = ax.imshow(attn_5, cmap='Reds', vmin=0)
for i in range(grid_size):
    for j in range(grid_size):
        idx = i * grid_size + j
        val = attn_5[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_title('Attention do Patch 5 (objeto)\n(foca em outros patches do objeto!)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.axis('off')

# 3. Attention do patch 0 (fundo)
ax = axes[2]
attn_0 = attn_weights[0].reshape(grid_size, grid_size)
im = ax.imshow(attn_0, cmap='Blues', vmin=0)
for i in range(grid_size):
    for j in range(grid_size):
        idx = i * grid_size + j
        val = attn_0[i, j]
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=9, fontweight='bold')
ax.set_title('Attention do Patch 0 (fundo)\n(foca em outros patches do fundo!)', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.axis('off')

plt.suptitle('Self-Attention agrupa patches semanticamente similares', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/attention_patterns.png', dpi=100, bbox_inches='tight')
plt.show()

# Quantificar
attn_obj_to_obj = sum(attn_weights[5][i] for i in object_patches)
attn_obj_to_bg = sum(attn_weights[5][i] for i in range(16) if i not in object_patches)
print(f'Patch 5 (objeto):')
print(f'  Atencao para OBJETO: {attn_obj_to_obj:.2%}')
print(f'  Atencao para FUNDO:  {attn_obj_to_bg:.2%}')

attn_bg_to_obj = sum(attn_weights[0][i] for i in object_patches)
attn_bg_to_bg = sum(attn_weights[0][i] for i in range(16) if i not in object_patches)
print(f'Patch 0 (fundo):')
print(f'  Atencao para OBJETO: {attn_bg_to_obj:.2%}')
print(f'  Atencao para FUNDO:  {attn_bg_to_bg:.2%}')
print()
print('Self-attention naturalmente agrupa patches do mesmo objeto!')
print('Isso emerge SEM supervisao explicita de segmentacao.')

### O que observar sobre Data Augmentation para ViTs

ViTs precisam de augmentation mais agressiva que CNNs:
- **RandAugment:** combinacoes aleatorias de transformacoes
- **Mixup/CutMix:** interpolar imagens e labels (essencial para ViT)
- **Random Erasing:** mascarar patches aleatorios (similar a dropout)

Sem augmentation forte, ViT overfita rapidamente em datasets < 1M imagens.

### O que concluir sobre Efficiency vs Performance em ViTs

Existe um espectro de trade-offs:
- **ViT-Tiny (5.7M params):** rapido, accuracy menor, bom para mobile
- **ViT-Base (86M params):** equilibrado, padrao para pesquisa
- **ViT-Large (304M params):** melhor accuracy, mas 4x mais lento
- **ViT-Huge (632M params):** SOTA em benchmarks, impraticavel para deploy

Na pratica, ViT-Base com bom pre-training supera ViT-Large sem pre-training.

### Conexao com outros notebooks sobre Regularizacao

As tecnicas de augmentation para ViT conectam com 4_1 (regularizacao) e 4_2
(dropout). CutMix e Mixup sao formas de regularizacao de dados, nao de pesos.
ViTs tambem usam Stochastic Depth (dropout de camadas inteiras).

### O que observar sobre Interpretabilidade de ViTs

Attention maps de ViTs sao mais interpretaveis que feature maps de CNNs:
- Cada head de attention "olha" para diferentes partes da imagem
- Heads das primeiras camadas: focam em textura local
- Heads das ultimas camadas: focam em semantica global
- CLS token das ultimas camadas: resume a imagem inteira

### O que concluir sobre Attention como Explicabilidade

Attention maps NAO sao explicacoes causais perfeitas, mas sao uteis para debugging.
Se o modelo erra, voce pode ver PARA ONDE ele estava olhando. Em aplicacoes
medicas, isso e valioso: "o modelo diagnosticou cancer porque olhou para ESTA regiao".

### Conexao com outros notebooks sobre Interpretabilidade

Interpretabilidade de ViTs conecta com 4_3 (interpretabilidade de modelos).
Grad-CAM funciona em CNNs, Attention Rollout funciona em ViTs. Ambos tentam
responder: "por que o modelo tomou essa decisao?"

### Conexao com outros notebooks sobre Ensemble e Model Selection

Na pratica, ensembles de ViT + CNN frequentemente superam ambos isolados (2_4 ensembles).
ViT captura relacoes globais, CNN captura texturas locais -- sao complementares.

## 8. Erros Comuns e Armadilhas

### Erro 1: Esquecer Positional Encoding

**O que acontece:** ViT sem positional embeddings nao entende posicao dos patches.
A imagem vira um "saco de patches" embaralhado.

**Por que em ML:** Transformers sao permutation-invariant por design.
Sem positional encoding, [patch_1, patch_2, ...] == [patch_5, patch_1, ...].

**Solucao:** Sempre incluir positional embeddings (learnable ou sinusoidal).

### Erro 2: Batch Size Muito Pequeno

**O que acontece:** ViT com batch=32 nao converge bem.

**Por que em ML:** ViT depende de BatchNorm/LayerNorm e gradientes estaveis.
Batches pequenos produzem estimativas ruidosas das estatisticas.

**Solucao:** Batch >= 256. Se GPU nao aguenta, usar gradient accumulation.

### Erro 3: Learning Rate Muito Alto

**O que acontece:** ViT com lr=0.01 (padrao CNN) diverge.

**Por que em ML:** Attention weights sao multiplicativas (Q*K). Gradientes
se amplificam mais facilmente que em convolucoes.

**Solucao:** lr=0.001 ou menor. AdamW e o otimizador padrao para ViT.

### Erro 4: Nao Usar Warm-Up

**O que acontece:** Treinamento instavel nos primeiros epochs.

**Por que em ML:** Attention weights aleatorias geram gradientes erraticos.
Warm-up permite que o modelo "encontre" uma regiao estavel primeiro.

**Solucao:** Linear warm-up por 5-20 epochs, depois cosine decay.

### Erro 5: Treinar ViT from Scratch em Dataset Pequeno

**O que acontece:** Accuracy muito pior que CNN equivalente.

**Por que em ML:** ViT tem MENOS inductive bias que CNN. Sem pre-training,
precisa aprender localidade e hierarquia do zero -- requer muitos dados.

**Solucao:** SEMPRE usar transfer learning. ViT pre-treinado em ImageNet-21K
+ fine-tune no seu dataset.

### Erro 6: Patch Size Errado

**O que acontece:** patch_size=8 com imagem 224x224 gera 784 patches.
Attention O(784^2) = O(614K) por camada. Memoria e tempo explodem.

**Por que em ML:** Custo quadratico em numero de patches.

**Solucao:** patch_size=16 e padrao (196 patches). So use patches menores
se tem GPU com memoria suficiente.

### O que observar sobre Computational Cost de Transformers

O custo de self-attention e O(N^2 * d) onde N = numero de tokens:
- ViT-B (patch 16, 224px): 196 tokens -> ~38K pares
- ViT-B (patch 16, 384px): 576 tokens -> ~331K pares (8.7x mais!)
- ViT-B (patch 8, 224px): 784 tokens -> ~614K pares (16x mais!)

Resolucao e patch size tem impacto QUADRATICO no custo. Por isso Swin
e variantes com attention local sao preferidos para imagens de alta resolucao.

### O que concluir sobre a Convergencia CNN-Transformer

A tendencia recente (ConvNeXt, 2022) mostra que CNNs com "receitas modernas"
(GELU, LayerNorm, patches grandes, etc.) competem com ViTs.
E ViTs com inductive biases locais (Swin) parecem CNNs hierarquicas.
Os dois mundos estao convergindo -- a arquitetura especifica importa menos
que o pre-training e o volume de dados.

### Conexao com outros notebooks sobre Otimizacao

O custo computacional de Transformers conecta com 4_5 (aceleracao hardware)
e 4_4 (mixed precision). Tecnicas como FlashAttention, gradient checkpointing
e mixed precision sao ESSENCIAIS para treinar ViTs em GPUs consumer.

### O que observar sobre Multimodalidade como Tendencia

A tendencia clara e modelos que entendem MULTIPLOS tipos de dados:
- CLIP: visao + texto
- ImageBind: visao + texto + audio + depth + thermal + IMU
- GPT-4V / Gemini: visao + texto + codigo + raciocinio
- Sora: visao + texto -> video

A base de tudo sao Transformers processando tokens de diferentes modalidades.

### O que concluir sobre o Futuro de Visao Computacional

Visao computacional esta se fundindo com NLP e outras modalidades.
O "computer vision specialist" do futuro precisa entender Transformers,
contrastive learning, e modelos multimodais -- nao apenas CNNs e data augmentation.

### Conexao com outros notebooks sobre o Curriculum Completo

Este notebook (ViTs) conecta todos os conceitos do modulo 5A:
- 5A_1 (CNNs): a base que ViTs melhoram/substituem
- 5A_2 (classificacao): transfer learning agora e com ViTs pre-treinados
- 5A_3 (deteccao): Swin e DINO revolucionaram deteccao
- 5A_4 (segmentacao): SAM e o estado da arte em segmentacao zero-shot

## Resumo e Proximos Passos

### Hierarquia de Conceitos

```
Vision Transformers
|
|-- Fundamentos
|   |-- Self-Attention (Q, K, V)
|   |-- Patch Embedding (imagem -> tokens)
|   |-- Positional Encoding (posicao dos patches)
|   +-- Transformer Encoder (attention + FFN + residual)
|
|-- Variantes de ViT
|   |-- DeiT (distillation, menos dados)
|   |-- Swin (shifted windows, O(N) linear)
|   +-- MAE (masked autoencoder, self-supervised)
|
|-- Self-Supervised Learning
|   |-- Contrastive (SimCLR, MoCo, BYOL)
|   |-- Masked (MAE, BEiT)
|   +-- Self-Distillation (DINO)
|
|-- Vision-Language
|   |-- CLIP (image-text alignment)
|   |-- Zero-shot classification
|   +-- Base para geracao (DALL-E, Stable Diffusion)
|
+-- Foundation Models
    |-- SAM (segmentacao universal)
    |-- DINOv2 (features universais)
    +-- Multimodais (GPT-4V, Gemini)
```

### Conexoes entre Notebooks

| Conceito deste notebook | Conecta com | Relacao |
|------------------------|-------------|---------|
| Self-Attention | 5A_1 (CNNs) | Receptive field global vs local |
| Transfer Learning ViT | 5A_2 (classificacao) | Pre-training massivo |
| Swin para deteccao | 5A_3 (deteccao) | FPN hierarquico |
| SAM | 5A_4 (segmentacao) | Segmentacao zero-shot |
| Scaling laws | 4_5 (hardware) | Mais compute = melhor ViT |
| Inductive bias | 2_1 (bias-variance) | Menos bias = mais dados |
| Self-supervised | 3_3 (clustering) | Aprender estrutura sem labels |

### Checklist de Competencias

- [ ] Entendo como self-attention funciona (Q, K, V, scores, weights)
- [ ] Sei explicar patch embedding e positional encoding
- [ ] Conhego as diferencas entre ViT, DeiT, Swin e MAE
- [ ] Entendo contrastive vs masked self-supervised learning
- [ ] Sei como CLIP faz zero-shot classification
- [ ] Conhego os foundation models (SAM, DINOv2)
- [ ] Sei quando usar ViT vs CNN (trade-off dados/inductive bias)
- [ ] Entendo os erros comuns ao treinar ViTs

### Proximos Passos

- **5B_1:** Transformers para NLP (a origem de tudo que vimos aqui)
- **5C_1:** Modelos Generativos (DALL-E, Stable Diffusion usam ViT + CLIP)
- **Pratica:** Fine-tunar ViT pre-treinado no seu dataset (HuggingFace transformers)